In [2]:
# ===== SETTINGS: the only cell to edit =====
VIDEO_DATASET = 'YOUR-VIDEO-DATASET'   # slug of your Kaggle dataset; also put it in kernel-metadata.json
MODE  = 'single'      # 'single' = one video (with preview), 'batch' = every video in the dataset
VIDEO = ''            # single mode: file name, e.g. 'my clip.mp4'; '' = first video alphabetically
TEXTS = """
"""                   # watermark strings you know, one per line; leave empty to rely on discovery
DISCOVER = True       # also look for repeated text you did not list (confident finds are removed,
                      # uncertain ones are only reported as "needs review")
ENCODER = 'auto'      # 'auto' = NVENC on the T4 (fast), 'x264' = CPU (keeps fine grain, ~5x slower)

In [3]:
import glob, json, os, shutil, subprocess
import cv2, numpy as np
SRC = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/wmr-*/wmr/__init__.py', recursive=True)[0]))
ROOT = '/kaggle/working/' + os.path.basename(SRC)          # e.g. wmr-0.3.0
shutil.copytree(SRC, ROOT, dirs_exist_ok=True)
IN_DIR = next(b for b, d, f in os.walk('/kaggle/input', followlinks=True)
              if VIDEO_DATASET in b and any(x.lower().endswith(('.mp4', '.mov', '.mkv', '.m4v', '.webm', '.avi')) for x in f))
videos = sorted(f for f in os.listdir(IN_DIR) if f.lower().endswith(('.mp4', '.mov', '.mkv', '.m4v', '.webm', '.avi')))
print(os.path.basename(ROOT), '| videos in', IN_DIR, ':', len(videos), videos[:10])
OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)
open('/kaggle/working/texts.txt', 'w').write(TEXTS.strip() + '\n')
FLAGS = ['--texts', '/kaggle/working/texts.txt', '--encoder', ENCODER] + (['--discover'] if DISCOVER else [])

StopIteration: 

In [ ]:
!bash /kaggle/working/wmr-*/scripts/setup_kaggle.sh

In [ ]:
# ===== run =====
if MODE == 'single':
    name = VIDEO or videos[0]
    stem = os.path.splitext(name)[0]
    cmd = ['python', '-m', 'wmr', 'run', os.path.join(IN_DIR, name), '-o', f'{OUT}/{stem}_clean.mp4',
           '--preview', f'{OUT}/{stem}_preview.mp4'] + FLAGS
else:
    cmd = ['python', '-m', 'wmr', 'batch', IN_DIR, '-o', OUT, '--gpus', '0,1'] + FLAGS
print(' '.join(cmd))
subprocess.run(cmd)

In [ ]:
# ===== evaluate: summary per video + before/after stills =====
def still(path, t):
    c = cv2.VideoCapture(path); c.set(cv2.CAP_PROP_POS_MSEC, t * 1000); ok, f = c.read(); c.release()
    return f if ok else None
for rep in sorted(glob.glob(f'{OUT}/*.report.json')):
    r = json.load(open(rep)); v = r.get('verify', {})
    tr = json.load(open(r['tracks']))
    print('\n==', os.path.basename(r['video']), '->', os.path.basename(r['output']))
    for t in tr['tracks']:
        print(f"   {'REMOVED' if t['enabled'] else 'needs review'}: {t['label']!r} ({t['source']}, {t['motion']}, "
              f"tier {t['tier']}, frames {t['first']}-{t['last']}) {t.get('note', '')}")
    print(f"   leftover check: {v.get('leak_frames', '-')} of {v.get('checked_frames', '-')} sampled frames still readable")
    print(f"   size: source {os.path.getsize(r['video']) / 1e6:.1f} MB, output {os.path.getsize(r['output']) / 1e6:.1f} MB")
    dur = tr['info']['duration']
    rows = []
    for t in (dur * 0.2, dur * 0.5, dur * 0.8):
        a, b = still(r['video'], t), still(r['output'], t)
        if a is not None and b is not None:
            rows.append(np.hstack([a, b]))
    if rows:
        img = np.vstack(rows); s = min(1.0, 1600 / img.shape[1])
        cv2.imwrite(os.path.splitext(r['output'])[0] + '_stills.jpg', cv2.resize(img, None, fx=s, fy=s), [cv2.IMWRITE_JPEG_QUALITY, 90])

In [ ]:
shutil.rmtree(ROOT, ignore_errors=True)   # keep the downloadable output to results only
!ls -la /kaggle/working/out